In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import kagglehub
import shutil

# Download dataset
path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Downloaded to:", path)

# Destination in Google Drive
destination = "/content/drive/MyDrive/fake-and-real-news-dataset"

# Copy the entire dataset folder
shutil.copytree(path, destination, dirs_exist_ok=True)

print("Saved to:", destination)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Downloaded to: /kaggle/input/fake-and-real-news-dataset
Saved to: /content/drive/MyDrive/fake-and-real-news-dataset


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report
import pickle

fake = pd.read_csv('/content/drive/MyDrive/fake-and-real-news-dataset/Fake.csv')
real = pd.read_csv('/content/drive/MyDrive/fake-and-real-news-dataset/True.csv')
fake['label'] = 0
real['label'] = 1
df = pd.concat([fake, real]).sample(frac=1, random_state=42).reset_index(drop=True)
df['content'] = df['title'] + " " + df['text']

X_train, X_test, y_train, y_test = train_test_split(
    df['content'], df['label'], test_size=0.2, random_state=42
)

tfidf = TfidfVectorizer(stop_words='english', max_df=0.7, ngram_range=(1,2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

model = PassiveAggressiveClassifier(max_iter=50)
model.fit(X_train_tfidf, y_train)

preds = model.predict(X_test_tfidf)
print(accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

0.9948775055679288
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      4710
           1       0.99      1.00      0.99      4270

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [5]:
with open('fake_news_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)